In [1]:
"""
Descarga de clasificación del suelo para análisis de construcción
planta de biometano — Provincia de Huesca

Fuentes:
  1. ICEAragón WFS (SIUa) — clasificación urbanística
  2. Red Natura 2000 (MITECO) — ZEC, ZEPA — zip shapefile oficial
  3. ENP (MITECO) — Parques, Reservas Naturales — WFS ICEAragón

Output: clasificacion_suelo_huesca.gpkg

NOTA sobre clasificacion_idearagon:
  La capa SIUa:clasificacion_idearagon NO está disponible vía GetFeature JSON
  en el endpoint SIUa_WMS. Se prueban variantes de nombre y parámetros.
  Si todo falla, descarga manual en: https://idearagon.aragon.es/SIUa/descargas
"""

import geopandas as gpd
import requests
import zipfile
import tempfile
import io
import pandas as pd
from pathlib import Path
from shapely.ops import unary_union

# ─────────────────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────────────────
DELIM_DIR      = Path("../../data/raw/delimitations")
OUTPUT_DIR     = Path("../../data/raw/06_clasificacion_suelo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GEOJSON_HUESCA = DELIM_DIR / "Huesca_Delimitacion.geojson"
OUTPUT_GPKG    = OUTPUT_DIR / "clasificacion_suelo_huesca.gpkg"

CRS_UTM    = "EPSG:25830"
BBOX_WGS84 = (-0.934168, 41.347806, 0.771832, 42.921806)  # minx,miny,maxx,maxy
HEADERS    = {"User-Agent": "biometano-huesca-research/1.0 (academic)"}
WFS_SIUA = "https://icearagon.aragon.es/geoserver/wfs"


def cargar_huesca():
    huesca = gpd.read_file(GEOJSON_HUESCA).to_crs(CRS_UTM)
    return unary_union(huesca.geometry)


# ─────────────────────────────────────────────────────────
# 1. ICEAragón SIUa — CLASIFICACIÓN URBANÍSTICA
# ─────────────────────────────────────────────────────────
def descargar_clasificacion_urbanistica():
    print("Descargando clasificación urbanística (ICEAragón SIUa)...")

    WFS_URL = "https://icearagon.aragon.es/geoserver/wfs"

    # BBOX de Huesca en EPSG:25830 (UTM zona 30N)
    # minx, miny, maxx, maxy
    BBOX_UTM = (585000, 4580000, 790000, 4760000)

    params = {
        "SERVICE":      "WFS",
        "VERSION":      "2.0.0",
        "REQUEST":      "GetFeature",
        "TYPENAMES":    "SIUa:clasificacion_idearagon",
        "OUTPUTFORMAT": "application/json",
        "SRSNAME":      "EPSG:25830",
        "BBOX":         f"{BBOX_UTM[0]},{BBOX_UTM[1]},{BBOX_UTM[2]},{BBOX_UTM[3]},EPSG:25830",
    }
    try:
        import io
        r = requests.get(WFS_URL, params=params, timeout=300, headers=HEADERS)
        r.raise_for_status()

        if b"ExceptionReport" in r.content[:500]:
            print(f"  ExceptionReport:\n{r.text}")
            return None

        gdf = gpd.read_file(io.BytesIO(r.content))

        if gdf is None or len(gdf) == 0:
            print("  Respuesta vacía")
            return None

        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:25830")
        gdf = gdf.to_crs(CRS_UTM)
        print(f"  Total: {len(gdf)} polígonos")
        col_cls = next((c for c in gdf.columns
                        if any(k in c.lower() for k in
                               ["clasif", "tipo", "clase", "uso"])), None)
        if col_cls:
            print(f"  Valores '{col_cls}': {gdf[col_cls].value_counts().to_dict()}")
        return gdf

    except requests.HTTPError as e:
        print(f"  Error HTTP {e.response.status_code}:\n{e.response.text[:800]}")
    except Exception as e:
        print(f"  Error: {e}")

    print("  Sin datos. Descarga manual:")
    print("  → https://idearagon.aragon.es/SIUa/descargas  (seleccionar Huesca, formato GPKG)")
    return None


# ─────────────────────────────────────────────────────────
# 2. RED NATURA 2000 — ZIP SHAPEFILE OFICIAL (MITECO)
# ─────────────────────────────────────────────────────────
def descargar_natura2000():
    """
    Descarga ZEC y ZEPA desde el zip oficial de MITECO.
    Extrae a carpeta temporal y lee el campo de tipo desde los atributos.
    """
    print("Descargando Red Natura 2000 (MITECO zip)...")

    URL_ZIP = ("https://www.miteco.gob.es/content/dam/miteco/es/biodiversidad"
               "/servicios/banco-datos-naturaleza/3-rn2000/rn2000-shp.zip")

    try:
        r = requests.get(URL_ZIP, timeout=180, headers=HEADERS)
        r.raise_for_status()

        h_union = cargar_huesca()

        with tempfile.TemporaryDirectory() as tmpdir:
            zipfile.ZipFile(io.BytesIO(r.content)).extractall(tmpdir)
            shapefiles = list(Path(tmpdir).rglob("*.shp"))
            print(f"  Shapefiles: {[s.name for s in shapefiles]}")

            gdfs = []
            for shp in shapefiles:
                nombre_lower = shp.stem.lower()
                if "_mac_" in nombre_lower or nombre_lower.endswith("_mac"):
                    print(f"  Saltando Macaronesia: {shp.name}")
                    continue
                try:
                    gdf = gpd.read_file(shp).to_crs(CRS_UTM)
                    gdf_h = gdf[gdf.geometry.intersects(h_union)].copy()
                    if len(gdf_h) == 0:
                        continue

                    col_tipo = next((c for c in gdf_h.columns
                                     if c.upper() in ["SITETYPE", "TIPO", "TYPE",
                                                       "SPTYPE", "SITE_TYPE"]), None)
                    if col_tipo:
                        print(f"  Tipos encontrados: {gdf_h[col_tipo].value_counts().to_dict()}")
                        def mapear_tipo(t):
                            t = str(t).upper()
                            if "SPA" in t or "ZEPA" in t:  return "ZEPA"
                            if "SAC" in t or "ZEC"  in t:  return "ZEC"
                            if "SCI" in t or "LIC"  in t:  return "LIC/SCI"
                            return t
                        gdf_h["tipo_proteccion"] = gdf_h[col_tipo].apply(mapear_tipo)
                    else:
                        nombre = shp.stem.upper()
                        gdf_h["tipo_proteccion"] = (
                            "ZEPA" if "ZEPA" in nombre or "SPA" in nombre else
                            "ZEC"  if "ZEC"  in nombre or "SAC" in nombre else
                            "LIC/SCI"
                        )

                    gdfs.append(gdf_h)
                    resumen = gdf_h["tipo_proteccion"].value_counts().to_dict()
                    print(f"  {shp.name}: {len(gdf_h)} zonas en Huesca → {resumen}")

                except Exception as e:
                    print(f"  Error leyendo {shp.name}: {e}")

        if gdfs:
            return gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=CRS_UTM)

    except Exception as e:
        print(f"  Descarga falló: {e}")

    print("  Descarga manual Natura 2000:")
    print("  → https://www.miteco.gob.es/es/cartografia-y-sig/ide/descargas/biodiversidad/rn2000.html")
    return None


# ─────────────────────────────────────────────────────────
# 3. ENP — WFS ICEAragón (SIUa:ENP)  ← igual que el original
# ─────────────────────────────────────────────────────────
def descargar_enp():
    """
    Descarga Espacios Naturales Protegidos desde el WFS de ICEAragón.
    IMPORTANTE: usa exactamente los mismos parámetros que funcionaban
    en el script original (VERSION 2.0.0, TYPENAMES SIUa:ENP, sin BBOX con CRS).
    """
    print("Descargando Espacios Naturales Protegidos (ICEAragón SIUa:ENP)...")

    minx, miny, maxx, maxy = BBOX_WGS84
    params = {
        "SERVICE":      "WFS",
        "VERSION":      "2.0.0",
        "REQUEST":      "GetFeature",
        "TYPENAMES":    "SIUa:ENP",
        "BBOX":         f"{minx},{miny},{maxx},{maxy},EPSG:4326",
        "SRSNAME":      "EPSG:4326",
        "OUTPUTFORMAT": "application/json",
        "COUNT":        "2000",
    }
    try:
        r = requests.get(WFS_SIUA, params=params, timeout=60, headers=HEADERS)
        features = r.json().get("features", [])
        if features:
            gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326").to_crs(CRS_UTM)
            print(f"  {len(gdf)} espacios protegidos")

            col_tipo = next((c for c in gdf.columns
                             if any(k in c.lower() for k in ["tipo", "type", "fig"])), None)
            if col_tipo:
                print(f"  Tipos: {gdf[col_tipo].value_counts().to_dict()}")
            return gdf
        else:
            print("  Sin features")
    except Exception as e:
        print(f"  Error: {e}")

    print("  Descarga manual ENP:")
    print("  → https://www.miteco.gob.es/es/cartografia-y-sig/ide/descargas/biodiversidad/enp.html")
    return None


# ─────────────────────────────────────────────────────────
# GUARDAR
# ─────────────────────────────────────────────────────────
def guardar(gdf_urbanistico, gdf_natura, gdf_enp):
    print()
    capas = {
        "clasificacion_urbanistica": gdf_urbanistico,
        "natura2000":                gdf_natura,
        "enp":                       gdf_enp,
    }
    for nombre, gdf in capas.items():
        if gdf is not None and len(gdf) > 0:
            gdf.to_file(OUTPUT_GPKG, driver="GPKG", layer=nombre)
            print(f"  '{nombre}': {len(gdf)} entidades → {OUTPUT_GPKG}")
        else:
            print(f"  '{nombre}': sin datos, no guardada")


# ─────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────
if __name__ == "__main__":

    print("\n" + "=" * 60)
    print("  CLASIFICACIÓN SUELO HUESCA — DESCARGA")
    print("=" * 60 + "\n")

    gdf_urbanistico = descargar_clasificacion_urbanistica()
    gdf_natura      = descargar_natura2000()
    gdf_enp         = descargar_enp()

    guardar(gdf_urbanistico, gdf_natura, gdf_enp)

    print("\nHecho. Capas en:", OUTPUT_GPKG)

    # ─────────────────────────────────────────────────────────
# GUARDAR CLASIFICACIÓN URBANÍSTICA EN ARCHIVOS
# ─────────────────────────────────────────────────────────

import geopandas as gpd

GPKG = Path("../../data/raw/06_clasificacion_suelo") / "clasificacion_suelo_huesca.gpkg"

# Guardar en el mismo GeoPackage que ya tienes (añade la capa)
gdf_urbanistico.to_file(GPKG, driver="GPKG", layer="clasificacion_urbanistica")
print(f"  'clasificacion_urbanistica': {len(gdf_urbanistico)} entidades → {GPKG}")


  CLASIFICACIÓN SUELO HUESCA — DESCARGA

Descargando clasificación urbanística (ICEAragón SIUa)...


  Total: 33527 polígonos
  Valores 'clasesiose': {'NULL': 393, 'SU-C': 180, 'SNU-G': 100, 'SU': 88, 'SNU-C': 1, 'SNU-E': 1}
Descargando Red Natura 2000 (MITECO zip)...


  Shapefiles: ['Es_Lic_SCI_Zepa_SPA_Medalpatl_202412.shp', 'Es_Lic_SCI_Zepa_SPA_Mac_202412.shp']


  Tipos encontrados: {'B': 66, 'A': 24, 'C': 12}
  Es_Lic_SCI_Zepa_SPA_Medalpatl_202412.shp: 102 zonas en Huesca → {'B': 66, 'A': 24, 'C': 12}
  Saltando Macaronesia: Es_Lic_SCI_Zepa_SPA_Mac_202412.shp
Descargando Espacios Naturales Protegidos (ICEAragón SIUa:ENP)...


  17 espacios protegidos
  Tipos: {'EP': 17}



  'clasificacion_urbanistica': 33527 entidades → ../../data/raw/06_clasificacion_suelo/clasificacion_suelo_huesca.gpkg
  'natura2000': 102 entidades → ../../data/raw/06_clasificacion_suelo/clasificacion_suelo_huesca.gpkg
  'enp': 17 entidades → ../../data/raw/06_clasificacion_suelo/clasificacion_suelo_huesca.gpkg

Hecho. Capas en: ../../data/raw/06_clasificacion_suelo/clasificacion_suelo_huesca.gpkg


  'clasificacion_urbanistica': 33527 entidades → ../../data/raw/06_clasificacion_suelo/clasificacion_suelo_huesca.gpkg


In [2]:
import numpy as np
import geopandas as gpd
import pandas as pd
import pydeck as pdk
from pathlib import Path

RAW_DIR = Path("../../data/raw/06_clasificacion_suelo")
MAP_DIR = Path("../../data/map/06_clasificacion_suelo")
RAW_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

GPKG = RAW_DIR / "clasificacion_suelo_huesca.gpkg"
gdf = gpd.read_file(GPKG, layer="clasificacion_urbanistica")
gdf = gdf[gdf["cod_ine"].astype(str).str.startswith("22")].copy()  # ← aquí
gdf = gdf.to_crs("EPSG:4326")
gdf["geometry"] = gdf["geometry"].simplify(0.0003, preserve_topology=True)
gdf = gdf[gdf["clase"].notna()].copy()

CLASE_STYLE = {
    "SUC":   {"color": [52,  152, 219, 255]},   # azul vivo
    "SUNC":  {"color": [41,  128, 185, 255]},   # azul oscuro
    "SDUD":  {"color": [243, 156,  18, 255]},   # naranja vivo
    "SDUNC": {"color": [230, 126,  34, 255]},   # naranja oscuro
    "SENU":  {"color": [ 39, 174,  96, 140]},   # verde — más transparente
    "SENE":  {"color": [ 26, 188, 156, 140]},   # verde agua — más transparente
    "SEUND": {"color": [155,  89, 182, 255]},   # morado vivo
}
DEFAULT = {"color": [150, 150, 150, 150]}

gdf["fill_color"] = gdf["clase"].apply(
    lambda c: CLASE_STYLE.get(str(c).strip(), DEFAULT)["color"]
)

def geom_to_coords(geom):
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == "Polygon":
        return [list(geom.exterior.coords)]
    if geom.geom_type == "MultiPolygon":
        biggest = max(geom.geoms, key=lambda p: p.area)
        return [list(biggest.exterior.coords)]
    return None

gdf["coordinates"] = gdf["geometry"].apply(geom_to_coords)
gdf = gdf[gdf["coordinates"].notna()].copy()

df = gdf[["coordinates", "clase", "clase_s", "d_muni_ine", "uso_glob", "fill_color"]].copy()
df["clase"]      = df["clase"].fillna("—")
df["clase_s"]    = df["clase_s"].fillna("—")
df["d_muni_ine"] = df["d_muni_ine"].fillna("—")
df["uso_glob"]   = df["uso_glob"].fillna("—")

print(f"Polígonos: {len(df):,}")

layer = pdk.Layer(
    "PolygonLayer",
    data=df,
    get_polygon="coordinates",
    get_fill_color="fill_color",
    get_line_color=[60, 60, 60, 100],
    line_width_min_pixels=0.3,
    extruded=False,
    pickable=True,
    auto_highlight=True,
    wireframe=False,
)

view = pdk.ViewState(
    longitude=-0.08,
    latitude=42.10,
    zoom=8,
    min_zoom=6,
    max_zoom=14,
    pitch=0,
    bearing=0,
)

tooltip = {
    "html": """
        <b>Clase:</b> {clase}<br/>
        <b>Subtipo:</b> {clase_s}<br/>
        <b>Municipio:</b> {d_muni_ine}<br/>
        <b>Uso global:</b> {uso_glob}
    """,
    "style": {"backgroundColor": "#111", "color": "white",
              "fontSize": "12px", "padding": "6px"},
}

LEGEND_HTML = """
<div style="position:fixed;bottom:30px;left:20px;background:rgba(0,0,0,0.75);
            color:#fff;padding:12px 16px;border-radius:8px;font-size:12px;
            font-family:sans-serif;z-index:9999;line-height:2">
  <b>Clasificación del suelo — Huesca</b><br/>
  <span style="color:#3498db">██</span> SUC   — Urbano Consolidado<br/>
  <span style="color:#2980b9">██</span> SUNC  — Urbano No Consolidado<br/>
  <span style="color:#f39c12">██</span> SDUD  — Urbanizable Delimitado<br/>
  <span style="color:#e67e22">██</span> SDUNC — Urbanizable No Delimitado<br/>
  <span style="color:#27ae60">██</span> SENU  — No Urbanizable Genérico<br/>
  <span style="color:#1abc9c">██</span> SENE  — No Urbanizable Especial<br/>
  <span style="color:#9b59b6">██</span> SEUND — No Urbanizable Especial ND<br/>
</div>
"""

deck = pdk.Deck(
    layers=[layer],
    initial_view_state=view,
    tooltip=tooltip,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
)

output_html = MAP_DIR / "mapa_clasificacion_huesca.html"
html = deck.to_html(as_string=True)
html = html.replace("</body>", LEGEND_HTML + "\n</body>")
with open(output_html, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Abre: {output_html}")

Polígonos: 17,344


Abre: ../../data/map/06_clasificacion_suelo/mapa_clasificacion_huesca.html


In [3]:
from pathlib import Path
GPKG = Path("../../data/raw/06_clasificacion_suelo") / "clasificacion_suelo_huesca.gpkg"
gdf_n = gpd.read_file(GPKG, layer="natura2000")
gdf_e = gpd.read_file(GPKG, layer="enp")
print("NATURA2000:", gdf_n.columns.tolist())
print(gdf_n.iloc[0])
print("\nENP:", gdf_e.columns.tolist())
print(gdf_e.iloc[0])

NATURA2000: ['OBJECTID', 'SITE_CODE', 'SITE_NAME', 'AC', 'TIPO', 'HECTAREAS', 'Shape_Leng', 'Shape_Area', 'tipo_proteccion', 'geometry']
OBJECTID                                                         552
SITE_CODE                                                  ES0000128
SITE_NAME                                       Sierra de San Miguel
AC                                                           NAVARRA
TIPO                                                               C
HECTAREAS                                                3113.515424
Shape_Leng                                              31037.673233
Shape_Area                                             31135154.2412
tipo_proteccion                                                    C
geometry           MULTIPOLYGON (((666902.5521 4739591.1709, 6669...
Name: 0, dtype: object

ENP: ['objectid', 'codigo', 'descripcio', 'ctipfigura', 'csubtip', 'cespecie', 'cestado', 'enp_id', 'superficie', 'cod_uicn', 'latitud', 'longtud', '

In [4]:
import numpy as np
import requests
import geopandas as gpd
import pandas as pd
import pydeck as pdk
from pathlib import Path

RAW_DIR = Path("../../data/raw/06_clasificacion_suelo")
MAP_DIR = Path("../../data/map/06_clasificacion_suelo")
RAW_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

# --- 1. Descargar padrón INE ---
print("Descargando padrón INE...")
url = "https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/2875?nult=1"
r = requests.get(url, timeout=60)
data = r.json()

registros = []
for item in data:
    nombre = item.get("Nombre", "")
    valor  = item.get("Data", [{}])[0].get("Valor", None)
    registros.append({"nombre": nombre, "poblacion": valor})

padron = pd.DataFrame(registros)

# Solo filas "Total" (sin Hombres/Mujeres) y sin la fila provincia
padron = padron[padron["nombre"].str.contains("Total habitantes")].copy()
padron = padron[~padron["nombre"].str.startswith("Huesca.")].copy()  # quitar total provincia

# Extraer nombre municipio limpio
padron["municipio"] = padron["nombre"].str.split(".").str[0].str.strip().str.upper()
padron["poblacion"] = pd.to_numeric(padron["poblacion"], errors="coerce").fillna(0).astype(int)
padron = padron[["municipio", "poblacion"]].drop_duplicates()
print(f"Municipios con población: {len(padron)}")

# --- 2. Buffer por población según normativa Aragón ---
def buffer_por_poblacion(pop):
    if pop < 3000: return 1000   # < 500 y 500-3000 hab → mismo valor: 1.000 m
    else:          return 2000  # > 3.000 hab → 1.500 m

GPKG = RAW_DIR / "clasificacion_suelo_huesca.gpkg"
gdf_raw = gpd.read_file(GPKG, layer="clasificacion_urbanistica")
gdf_raw = gdf_raw[gdf_raw["cod_ine"].astype(str).str.startswith("22")]
nucleos = gdf_raw[gdf_raw["clase"].isin(["SUC", "SUNC"])].copy()
nucleos_diss = nucleos.dissolve(by="d_muni_ine").reset_index()
nucleos_diss["municipio_key"] = nucleos_diss["d_muni_ine"].str.strip().str.upper()

# Unir con padrón
nucleos_diss = nucleos_diss.merge(padron, left_on="municipio_key",
                                   right_on="municipio", how="left")
nucleos_diss["poblacion"] = nucleos_diss["poblacion"].fillna(0).astype(int)
nucleos_diss["buffer_m"]  = nucleos_diss["poblacion"].apply(buffer_por_poblacion)

print(f"Municipios sin match padrón: {(nucleos_diss['poblacion']==0).sum()}")
print(nucleos_diss[["d_muni_ine", "poblacion", "buffer_m"]].sort_values("poblacion", ascending=False).head(10))

# Aplicar buffer individual
nucleos_diss["geometry"] = nucleos_diss.apply(
    lambda row: row.geometry.buffer(row["buffer_m"]), axis=1
)

# Guardar buffer en raw/ (geometría completa, EPSG:25830) — lo usa idoneidad_scoring_clustering
BUFFER_GPKG = RAW_DIR / "buffer_nucleos_urbanos.gpkg"
nucleos_diss.to_file(BUFFER_GPKG, driver="GPKG")
print(f"Guardado: {BUFFER_GPKG}")

nucleos_buffer = nucleos_diss.to_crs("EPSG:4326")
nucleos_buffer["geometry"] = nucleos_buffer["geometry"].simplify(0.001, preserve_topology=True)

# --- 3. Clasificación urbanística ---
gdf = gpd.read_file(GPKG, layer="clasificacion_urbanistica")
gdf = gdf[gdf["cod_ine"].astype(str).str.startswith("22")].copy()
gdf = gdf.to_crs("EPSG:4326")
gdf["geometry"] = gdf["geometry"].simplify(0.0003, preserve_topology=True)
gdf = gdf[gdf["clase"].notna()].copy()

CLASE_STYLE = {
    "SUC":   {"color": [52,  152, 219, 255]},
    "SUNC":  {"color": [41,  128, 185, 255]},
    "SDUD":  {"color": [243, 156,  18, 255]},
    "SDUNC": {"color": [230, 126,  34, 255]},
    "SENU":  {"color": [ 39, 174,  96, 140]},
    "SENE":  {"color": [ 26, 188, 156, 140]},
    "SEUND": {"color": [155,  89, 182, 255]},
}
DEFAULT = {"color": [150, 150, 150, 150]}

gdf["fill_color"] = gdf["clase"].apply(
    lambda c: CLASE_STYLE.get(str(c).strip(), DEFAULT)["color"]
)

def geom_to_coords(geom):
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == "Polygon":
        return [list(geom.exterior.coords)]
    if geom.geom_type == "MultiPolygon":
        biggest = max(geom.geoms, key=lambda p: p.area)
        return [list(biggest.exterior.coords)]
    return None

gdf["coordinates"] = gdf["geometry"].apply(geom_to_coords)
gdf = gdf[gdf["coordinates"].notna()].copy()
df = gdf[["coordinates", "clase", "clase_s", "d_muni_ine", "uso_glob", "fill_color"]].copy()
for col in ["clase", "clase_s", "d_muni_ine", "uso_glob"]:
    df[col] = df[col].fillna("—")

# --- 5. Capas pydeck ---
layer_suelo = pdk.Layer(
    "PolygonLayer",
    data=df,
    get_polygon="coordinates",
    get_fill_color="fill_color",
    get_line_color=[60, 60, 60, 100],
    line_width_min_pixels=0.3,
    extruded=False,
    pickable=True,
    auto_highlight=True,
)

view = pdk.ViewState(
    longitude=-0.08,
    latitude=42.10,
    zoom=8,
    min_zoom=6,
    max_zoom=14,
    pitch=0,
    bearing=0,
)

tooltip = {
    "html": """
        <b>Clase:</b> {clase}<br/>
        <b>Subtipo:</b> {clase_s}<br/>
        <b>Municipio:</b> {d_muni_ine}<br/>
        <b>Uso global:</b> {uso_glob}<br/>
    """,
    "style": {"backgroundColor": "#111", "color": "white",
              "fontSize": "12px", "padding": "6px"},
}

LEGEND_HTML = """
<div style="position:fixed;bottom:30px;left:20px;background:rgba(0,0,0,0.75);
            color:#fff;padding:12px 16px;border-radius:8px;font-size:12px;
            font-family:sans-serif;z-index:9999;line-height:2">
  <b>Clasificación del suelo — Huesca</b><br/>
  <span style="color:#3498db">██</span> SUC   — Urbano Consolidado<br/>
  <span style="color:#2980b9">██</span> SUNC  — Urbano No Consolidado<br/>
  <span style="color:#f39c12">██</span> SDUD  — Urbanizable Delimitado<br/>
  <span style="color:#e67e22">██</span> SDUNC — Urbanizable No Delimitado<br/>
  <span style="color:#27ae60">██</span> SENU  — No Urbanizable Genérico<br/>
  <span style="color:#1abc9c">██</span> SENE  — No Urbanizable Especial<br/>
  <span style="color:#9b59b6">██</span> SEUND — No Urbanizable Especial ND<br/>
  <hr style="border-color:#444;margin:4px 0"/>
</div>
"""

deck = pdk.Deck(
    layers=[layer_suelo],
    initial_view_state=view,
    tooltip=tooltip,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
)

output_html = MAP_DIR / "mapa_clasificacion_buffer_poblacion_huesca.html"
html = deck.to_html(as_string=True)
html = html.replace("</body>", LEGEND_HTML + "\n</body>")
with open(output_html, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Abre: {output_html}")

Descargando padrón INE...


Municipios con población: 600


Municipios sin match padrón: 11
     d_muni_ine  poblacion  buffer_m
325      MONZÓN      18525      2000
110   BARBASTRO      17807      2000
246       FRAGA      15576      2000
278        JACA      14024      2000
143     BINÉFAR      10235      2000
396  SABIÑÁNIGO       9695      2000
326      MONZÓN       9459      2000
112   BARBASTRO       9085      2000
327      MONZÓN       9066      2000
111   BARBASTRO       8722      2000


Guardado: ../../data/raw/06_clasificacion_suelo/buffer_nucleos_urbanos.gpkg


Abre: ../../data/map/06_clasificacion_suelo/mapa_clasificacion_buffer_poblacion_huesca.html
